# Langham Hotel Management System
**Course:** BBIM502 – Introduction to Programming  
**Assessment:** 2 – Project  
**Purpose:** A Python console application to manage rooms, bookings, billing, and file records for Langham Hotels.


In [1]:
"""
--------------------------------------------------------
File        : Langham_HMS.ipynb
Author      : Jaco
Student ID  : 850009268
Description : Console-based Hotel Management System for
              Langham Hotels. Covers room management,
              guest bookings, billing, file I/O, and
              exception handling.
Date        : June 2026
Version     : 1.0
--------------------------------------------------------
"""

# Standard library imports needed for file operations and timestamps
import os
import sys
import datetime

# ---- Global storage -----------------------------------------------
# room_catalogue    : room_no -> {room_type, nightly_rate, availability}
# allocation_register  : room_no -> {name, id_number, stay_nights, arrival, charge}

room_catalogue   = {}   # all rooms known to the system
allocation_register = {}   # rooms currently occupied by guests

# File naming follows the required convention: LHMS_<StudentID>.txt
SID       = "850009268"
DATA_FILE = f"LHMS_{SID}.txt"

print("Setup complete")


Setup complete


In [2]:
# ==============================================================
# DISPLAY HELPERS
# ==============================================================

def banner(heading):
    """
    Draws a plain horizontal banner around a heading string.
    Built-in functions used: print(), str.upper(), str.center()

    Parameters
    ----------
    heading : str  Text to show inside the banner.
    """
    line = "-" * 70
    print(line)
    print(heading.upper().center(70))
    print(line)


def show_main_menu():
    """
    Prints the numbered application menu to the console.
    Built-in functions used: print()
    """
    print("\n" + "=" * 70)
    print("  LANGHAM HOTEL MANAGEMENT SYSTEM".center(70))
    print("=" * 70)
    options = [
        " 1  Add a Room",
        " 2  Delete a Room",
        " 3  Display Rooms Details",
        " 4  Allocate Rooms",
        " 5  Display Room Allocation Details",
        " 6  Billing and De-Allocation",
        " 7  Save Allocations to File",
        " 8  Load Allocations from File",
        " 9  Backup and Wipe File",
        " 0  Quit",
    ]
    for opt in options:
        print(opt)
    print("=" * 70)

print("Display helpers ready.")


Display helpers ready.


In [3]:
# ==============================================================
# OPTION 1 - ADD A ROOM
# ==============================================================

def add_room():
    """
    Registers a new room in the hotel system by collecting the
    room number, category, and nightly rate from keyboard input.

    Built-in functions : input(), print(), float(), str()
    Exceptions handled : ValueError   - rate entered as non-numeric text
                         EOFError     - input stream unexpectedly closed
                         TypeError    - mismatched types during comparison
    """
    banner("Add a Room")
    try:
        rm_no = input("Room number  : ").strip()

        # Reject a blank room number entry
        if rm_no == "":
            raise ValueError("Room number cannot be blank.")

        # Prevent overwriting data for an existing room
        if rm_no in room_catalogue:
            print(f"Room {rm_no} already exists in the system.")
            return

        category = input("Category (Single / Double / Suite): ").strip()
        rate_str = input("Nightly rate (NZD)                 : ").strip()

        # float() raises ValueError if rate_str contains letters
        nightly_rate = float(rate_str)

        if nightly_rate <= 0:
            raise ValueError("Nightly rate must be greater than zero.")

        room_catalogue[rm_no] = {
            "room_type"    : category,
            "nightly_rate" : nightly_rate,
            "availability" : "Available",
        }
        print(f"Room {rm_no} ({category}) added at ${nightly_rate:.2f} per night.")

    except ValueError as err:
        # Triggered when the user types letters where a number is expected
        print(f"[ValueError] {err}")
    except TypeError as err:
        # Triggered if an operation uses the wrong data type
        print(f"[TypeError] Unexpected type issue: {err}")
    except EOFError:
        # Triggered if input() receives no data (e.g. piped input ends early)
        print("[EOFError] Input ended before data was received.")

print("add_room() ready.")


add_room() ready.


In [4]:
# ==============================================================
# OPTION 2 - DELETE A ROOM
# ==============================================================

def delete_room():
    """
    Removes a room from the catalogue as long as it has no
    active booking assigned to it.

    Built-in functions : input(), print(), dict.pop()
    Exceptions handled : KeyError   - room number not found in catalogue
                         NameError  - undefined variable referenced
                         EOFError   - input stream closed unexpectedly
    """
    banner("Delete a Room")
    try:
        rm_no = input("Enter the room number to remove: ").strip()

        # Raise KeyError manually if the room is not listed
        if rm_no not in room_catalogue:
            raise KeyError(f"Room {rm_no} is not listed in the catalogue.")

        # Block deletion while the room has an active allocation
        if rm_no in allocation_register:
            print(f"Cannot delete Room {rm_no} - it has an active allocation. "
                  "Check out the guest first.")
            return

        removed_room = room_catalogue.pop(rm_no)
        print(f"Room {rm_no} ({removed_room['room_type']}) has been removed.")

    except KeyError as err:
        # Raised when trying to access a key that does not exist in a dict
        print(f"[KeyError] {err}")
    except NameError as err:
        # Raised when a variable is used before it has been defined
        print(f"[NameError] {err}")
    except EOFError:
        print("[EOFError] No input was received.")

print("delete_room() ready.")


delete_room() ready.


In [5]:
# ==============================================================
# OPTION 3 - SHOW ALL ROOMS
# ==============================================================

def show_all_rooms():
    """
    Prints a formatted table of every room in the catalogue
    with its type, rate, and current availability status.

    Built-in functions : print(), list(), len(), range(), f-strings
    Exceptions handled : IndexError  - index out of bounds on room list
                         TypeError   - formatting error caused by bad data type
    """
    banner("All Rooms")
    try:
        if len(room_catalogue) == 0:
            print("No rooms have been added yet.")
            return

        # Table header row
        print(f"{'No.':<10} {'Category':<16} {'Rate/Night':>12} {'Status':<12}")
        print("-" * 55)

        # Convert dict to list so we can use index-based iteration (IndexError demo)
        entries = list(room_catalogue.items())

        for idx in range(len(entries)):
            rm_no, details = entries[idx]   # IndexError if idx out of range
            if rm_no in allocation_register:
                current_status = "Occupied"
            else:
                current_status = details["availability"]
            print(f"{rm_no:<10} {details['room_type']:<16} "
                  f"${details['nightly_rate']:>11.2f} {current_status:<12}")

    except IndexError as err:
        # Raised when a sequence index is outside the valid range
        print(f"[IndexError] {err}")
    except TypeError as err:
        print(f"[TypeError] {err}")

print("show_all_rooms() ready.")


show_all_rooms() ready.


In [6]:
# ==============================================================
# OPTION 4 - ALLOCATE A ROOM
# ==============================================================

def allocate_room():
    """
    Records a new allocation for an available room and calculates
    the estimated total charge.

    Built-in functions : input(), print(), int(), str()
    Modules used       : datetime - records today as the check-in date
    Exceptions handled : ValueError    - nights entered as non-integer text
                         OverflowError - astronomically large stay duration
                         EOFError      - input stream closes mid-entry
    """
    banner("Allocate a Room")
    try:
        # Show rooms first so the user can pick one
        show_all_rooms()

        rm_no = input("\nRoom number to allocate : ").strip()

        if rm_no not in room_catalogue:
            print(f"Room {rm_no} does not exist.")
            return

        if rm_no in allocation_register:
            print(f"Room {rm_no} is already occupied.")
            return

        guest_full_name = input("Guest full name     : ").strip()
        guest_id_no     = input("Passport / ID no.   : ").strip()
        nights_str      = input("Number of nights    : ").strip()

        # int() raises ValueError if nights_str contains letters or decimals
        total_nights = int(nights_str)

        if total_nights <= 0:
            raise ValueError("Number of nights must be a positive whole number.")

        # Guard against values that would overflow float arithmetic
        estimated_bill = total_nights * room_catalogue[rm_no]["nightly_rate"]
        if estimated_bill > 1e15:
            raise OverflowError("Calculated charge exceeds the allowable numeric limit.")

        allocation_register[rm_no] = {
            "name"        : guest_full_name,
            "id_number"   : guest_id_no,
            "stay_nights" : total_nights,
            "arrival"     : str(datetime.date.today()),
            "charge"      : estimated_bill,
        }
        room_catalogue[rm_no]["availability"] = "Occupied"
        print(f"\nRoom {rm_no} booked for {guest_full_name} "
              f"({total_nights} night(s), est. ${estimated_bill:.2f}).")

    except ValueError as err:
        # Raised when the input cannot be converted to the expected numeric type
        print(f"[ValueError] {err}")
    except OverflowError as err:
        # Raised when a numeric result exceeds the maximum floating-point limit
        print(f"[OverflowError] {err}")
    except EOFError:
        print("[EOFError] Input stream ended unexpectedly.")

print("book_room() ready.")


book_room() ready.


In [7]:
# ==============================================================
# OPTION 5 - DISPLAY ROOM ALLOCATION DETAILS
# ==============================================================

def view_allocations():
    """
    Displays a summary table of every room that is currently allocated.

    Built-in functions : print(), f-strings
    Exceptions handled : NameError - variable used before it is defined
                         KeyError  - missing field inside an allocation record
    """
    banner("Current Allocations")
    try:
        if not allocation_register:
            print("There are no active allocations at the moment.")
            return

        # Column headers
        print(f"{'Room':<8} {'Guest Name':<22} {'ID No.':<14} "
              f"{'Nights':>7} {'Check-in':<13} {'Est. Bill':>11}")
        print("-" * 80)

        for rm_no, rec in allocation_register.items():
            print(f"{rm_no:<8} {rec['name']:<22} {rec['id_number']:<14} "
                  f"{rec['stay_nights']:>7} {rec['arrival']:<13} "
                  f"${rec['charge']:>10.2f}")

    except NameError as err:
        # Raised when a variable name cannot be found in the current scope
        print(f"[NameError] {err}")
    except KeyError as err:
        print(f"[KeyError] Missing field in allocation record: {err}")

print("view_allocations() ready.")


view_allocations() ready.


In [8]:
# ==============================================================
# OPTION 6 - BILLING AND DE-ALLOCATION
# ==============================================================

def checkout():
    """
    Generates an itemised invoice including NZ GST (15%) for a
    allocated room, then marks the room as available again.

    Built-in functions : input(), print(), float(), round()
    Modules used       : datetime - for the checkout date
    Exceptions handled : ZeroDivisionError - nights value is zero
                         ValueError        - bad numeric data in record
                         KeyError          - missing field in the allocation record
    """
    banner("Checkout and Bill")
    try:
        if not allocation_register:
            print("No active allocations to check out.")
            return

        view_allocations()
        rm_no = input("\nEnter room number to check out: ").strip()

        if rm_no not in allocation_register:
            print(f"No active allocation found for room {rm_no}.")
            return

        rec         = allocation_register[rm_no]
        stay_nights = rec["stay_nights"]
        rate        = room_catalogue[rm_no]["nightly_rate"]

        # Guard: stay_nights should never be zero, but check to prevent ZeroDivisionError
        if stay_nights == 0:
            raise ZeroDivisionError(
                "Cannot compute rate per night - stay nights recorded as zero.")

        subtotal    = stay_nights * rate
        gst         = round(subtotal * 0.15, 2)   # New Zealand GST rate
        grand_total = round(subtotal + gst, 2)

        # Print formatted invoice
        print()
        print("#" * 52)
        print("  LANGHAM HOTELS - TAX INVOICE".center(52))
        print("#" * 52)
        print(f"  Room          : {rm_no}")
        print(f"  Category      : {room_catalogue[rm_no]['room_type']}")
        print(f"  Guest         : {rec['name']}")
        print(f"  Passport/ID   : {rec['id_number']}")
        print(f"  Check-in      : {rec['arrival']}")
        print(f"  Check-out     : {datetime.date.today()}")
        print("  " + "-" * 48)
        print(f"  Nights stayed : {stay_nights}")
        print(f"  Rate per night: ${rate:.2f}")
        print(f"  Subtotal      : ${subtotal:.2f}")
        print(f"  GST (15%)     : ${gst:.2f}")
        print(f"  TOTAL DUE     : ${grand_total:.2f}")
        print("#" * 52)

        # Free the room for new bookings
        del allocation_register[rm_no]
        room_catalogue[rm_no]["availability"] = "Available"
        print(f"\nRoom {rm_no} is now available for new guests.")

    except ZeroDivisionError as err:
        # Raised when the denominator in an arithmetic expression equals zero
        print(f"[ZeroDivisionError] {err}")
    except ValueError as err:
        print(f"[ValueError] {err}")
    except KeyError as err:
        print(f"[KeyError] Missing data field: {err}")

print("checkout() ready.")


checkout() ready.


In [9]:
# ==============================================================
# OPTION 7 - SAVE ALLOCATIONS TO FILE
# ==============================================================

def save_to_file():
    """
    Writes the room catalogue and all active allocations to a text
    file. The file is created if it does not already exist.

    Built-in functions : open(), write(), close()
    Modules used       : datetime - report timestamp
                         os       - file path operations
    Exceptions handled : IOError           - file cannot be opened or written
                         FileNotFoundError - target directory path does not exist
    """
    banner("Save to File")
    fh = None   # file handle kept outside try so finally can always close it
    try:
        fh    = open(DATA_FILE, "w")   # 'w' creates file or overwrites existing
        stamp = datetime.datetime.now().strftime("%d %B %Y  %H:%M:%S")

        fh.write("LANGHAM HOTEL MANAGEMENT SYSTEM\n")
        fh.write(f"Report generated: {stamp}\n")
        fh.write("=" * 60 + "\n")

        # Section 1 - all rooms
        fh.write("\n--- ROOM CATALOGUE ---\n")
        fh.write(f"{'Room':<10}{'Type':<16}{'Rate':>10}{'Status':>12}\n")
        fh.write("-" * 50 + "\n")
        for rm_no, info in room_catalogue.items():
            if rm_no in allocation_register:
                st = "Occupied"
            else:
                st = info["availability"]
            fh.write(f"{rm_no:<10}{info['room_type']:<16}"
                     f"${info['nightly_rate']:>9.2f}{st:>12}\n")

        # Section 2 - active allocations
        fh.write("\n--- ACTIVE ALLOCATIONS ---\n")
        if allocation_register:
            fh.write(f"{'Room':<8}{'Guest':<22}{'Nights':>7}{'Charge':>12}\n")
            fh.write("-" * 52 + "\n")
            for rm_no, rec in allocation_register.items():
                fh.write(f"{rm_no:<8}{rec['name']:<22}"
                         f"{rec['stay_nights']:>7}${rec['charge']:>11.2f}\n")
        else:
            fh.write("No active allocations.\n")

        print(f"Data saved to '{DATA_FILE}'.")

    except FileNotFoundError as err:
        # Raised if the directory path for the file does not exist
        print(f"[FileNotFoundError] {err}")
    except IOError as err:
        # Raised when an I/O operation such as open() or write() fails
        print(f"[IOError] Could not write to file: {err}")
    finally:
        if fh:
            fh.close()
            print("File closed.")

print("save_to_file() ready.")


save_to_file() ready.


In [10]:
# ==============================================================
# OPTION 8 - LOAD ALLOCATIONS FROM FILE
# ==============================================================

def load_from_file():
    """
    Opens the data file and prints its full contents to screen.

    Built-in functions : open(), read(), print()
    Exceptions handled : FileNotFoundError - file has not been created yet
                         IOError           - file cannot be opened or read
    """
    banner("Load from File")
    fh = None
    try:
        fh       = open(DATA_FILE, "r")   # FileNotFoundError if file is missing
        contents = fh.read()

        if not contents.strip():
            print("The data file exists but is currently empty.")
        else:
            print(contents)

    except FileNotFoundError:
        # File not yet created - guide the user to save first
        print(f"[FileNotFoundError] '{DATA_FILE}' not found. "
              "Run Option 7 to save data first.")
    except IOError as err:
        print(f"[IOError] Could not read file: {err}")
    finally:
        if fh:
            fh.close()
            print("File closed.")

print("load_from_file() ready.")


load_from_file() ready.


In [11]:
# ==============================================================
# OPTION 9 - BACKUP AND WIPE FILE
# ==============================================================

def backup_and_wipe():
    """
    Copies all content from the main data file into a timestamped
    backup file, then clears the main file.

    Built-in functions : open(), read(), write(), close()
    Modules used       : datetime - timestamp used in the backup filename
                         os       - file system operations
    Exceptions handled : FileNotFoundError - source file does not exist
                         IOError           - read or write failure
    """
    banner("Backup and Wipe File")
    ts          = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    backup_name = f"LHMS_{SID}_Backup_{ts}.txt"

    src_fh  = None
    dest_fh = None
    try:
        # Step 1 - read the existing data file
        src_fh   = open(DATA_FILE, "r")
        snapshot = src_fh.read()
        src_fh.close()
        src_fh = None   # mark closed so finally does not double-close

        # Step 2 - append snapshot into the backup file
        dest_fh = open(backup_name, "a")   # 'a' appends without overwriting
        dest_fh.write(snapshot)
        dest_fh.close()
        dest_fh = None
        print(f"Backup written to '{backup_name}'.")

        # Step 3 - truncate the main file by opening it in write mode
        wipe_fh = open(DATA_FILE, "w")
        wipe_fh.close()
        print(f"'{DATA_FILE}' has been cleared.")

    except FileNotFoundError:
        print(f"[FileNotFoundError] '{DATA_FILE}' does not exist. "
              "Save data first using Option 7.")
    except IOError as err:
        print(f"[IOError] File operation failed: {err}")
    finally:
        if src_fh:
            src_fh.close()
        if dest_fh:
            dest_fh.close()

print("backup_and_wipe() ready.")


backup_and_wipe() ready.


In [12]:
# ==============================================================
# MAIN APPLICATION LOOP
# ==============================================================

def run_hotel_system():
    """
    Entry point. Displays the menu in a continuous loop until
    the user selects option 0 to quit the application.

    Built-in functions : input(), print(), int()
    Exceptions handled : ValueError        - non-integer menu choice entered
                         EOFError          - input stream closes unexpectedly
                         KeyboardInterrupt - user presses Ctrl+C
    """
    # Dispatch table maps menu numbers to their handler functions
    handlers = {
        1 : add_room,
        2 : delete_room,
        3 : show_all_rooms,
        4 : allocate_room,
        5 : view_allocations,
        6 : checkout,
        7 : save_to_file,
        8 : load_from_file,
        9 : backup_and_wipe,
    }

    while True:
        try:
            show_main_menu()
            raw_choice = input("Enter choice: ").strip()
            selection  = int(raw_choice)   # ValueError on non-integer input

            if selection == 0:
                banner("Thank you for using Langham Hotel Management System")
                print("Goodbye!\n")
                break
            elif selection in handlers:
                handlers[selection]()   # call the matching function
            else:
                print(f"Invalid choice '{selection}'. Please enter a number from 0 to 9.")

        except ValueError:
            # Raised when the user types something that cannot be converted to int
            print("[ValueError] Menu choice must be a whole number (0-9).")
        except EOFError:
            # Raised when the input stream ends before the user makes a choice
            print("\n[EOFError] Input stream ended. Closing system.")
            break
        except KeyboardInterrupt:
            # Raised when the user presses Ctrl+C
            print("\n[KeyboardInterrupt] Interrupted by user. Closing system.")
            break


# Start the application
run_hotel_system()



                    LANGHAM HOTEL MANAGEMENT SYSTEM                   
 1  Add a Room
 2  Delete a Room
 3  Display Rooms Details
 4  Allocate Rooms
 5  Display Room Allocation Details
 6  Billing and De-Allocation
 7  Save Allocations to File
 8  Load Allocations from File
 9  Backup and Wipe File
 0  Quit
[ValueError] Menu choice must be a whole number (0-9).

                    LANGHAM HOTEL MANAGEMENT SYSTEM                   
 1  Add a Room
 2  Delete a Room
 3  Display Rooms Details
 4  Allocate Rooms
 5  Display Room Allocation Details
 6  Billing and De-Allocation
 7  Save Allocations to File
 8  Load Allocations from File
 9  Backup and Wipe File
 0  Quit

[KeyboardInterrupt] Interrupted by user. Closing system.


## Task 2 – Error Detection and Exception Handling

### Six Errors Identified and Corrected

| # | Error Type | Location | Problem | Fix Applied |
|---|-----------|----------|---------|-------------|
| 1 | ValueError | `add_room()` – nightly rate | User could type letters (e.g. `abc`) where a number was needed, causing an unhandled crash | Wrapped `float()` in `try/except ValueError` with a clear error message |
| 2 | Logical | `add_room()` – duplicate check | The same room number could be added twice, silently overwriting existing room data | Added `if rm_no in room_catalogue` guard before saving the new entry |
| 3 | Runtime | `delete_room()` – occupied room | A room with an active booking could be deleted, leaving a dangling booking record | Added a check against `allocation_register` and blocked deletion if a booking exists |
| 4 | ValueError | `allocate_room()` – nights input | User could type `"two"` or a negative number, causing a crash or nonsensical booking | Used `int()` inside `try/except ValueError` and validated `total_nights > 0` |
| 5 | ZeroDivisionError | `checkout()` – billing | If `stay_nights` was recorded as `0`, the billing calculation would fail with an unhandled error | Added an explicit zero-check before billing with a descriptive `ZeroDivisionError` message |
| 6 | FileNotFoundError | `load_from_file()` and `backup_and_wipe()` | Reading or backing up the file before it was ever saved caused an unhandled crash | Wrapped `open()` in `try/except FileNotFoundError` with a user-friendly prompt to save first |


In [13]:
# ==============================================================
# TASK 2 - EXCEPTION HANDLING DEMONSTRATION
# Each of the 11 required exception types is triggered and caught
# in its own isolated try/except block using hotel-relevant context.
# This cell runs independently and does not affect the live system.
# ==============================================================

def run_exception_showcase():
    """
    Hotel-themed demonstration of all 11 required built-in exceptions.
    Each block uses a realistic hotel scenario to trigger the exception,
    then catches it with a descriptive message matching the assessment rubric.

    Exceptions covered:
        SyntaxError, ValueError, ZeroDivisionError, IndexError,
        NameError, TypeError, OverflowError, IOError, ImportError,
        EOFError, FileNotFoundError
    """
    print("=" * 65)
    print("  TASK 2 – EXCEPTION HANDLING SHOWCASE".center(65))
    print("=" * 65)

    # ------------------------------------------------------------------
    # 1. SyntaxError
    #    Scenario: the system tries to compile an incomplete command string
    # ------------------------------------------------------------------
    try:
        compile("while room_no :", "<hotel_cmd>", "exec")
    except SyntaxError as se:
        print(f"[1] SyntaxError       >> {se.msg} (line {se.lineno})")

    # ------------------------------------------------------------------
    # 2. ValueError
    #    Scenario: a receptionist types 'three' instead of '3' for nights
    # ------------------------------------------------------------------
    try:
        stay_duration = int("three")
    except ValueError as ve:
        print(f"[2] ValueError        >> {ve}")

    # ------------------------------------------------------------------
    # 3. ZeroDivisionError
    #    Scenario: compute average nightly spend but total_nights is zero
    # ------------------------------------------------------------------
    try:
        total_revenue = 850.00
        total_nights  = 0
        avg_per_night = total_revenue / total_nights
    except ZeroDivisionError as zd:
        print(f"[3] ZeroDivisionError >> {zd}")

    # ------------------------------------------------------------------
    # 4. IndexError
    #    Scenario: attempt to read the 10th room from a 3-room list
    # ------------------------------------------------------------------
    try:
        available_rooms = ["201", "305", "410"]
        selected = available_rooms[10]
    except IndexError as ie:
        print(f"[4] IndexError        >> {ie}")

    # ------------------------------------------------------------------
    # 5. NameError
    #    Scenario: reference a variable that was never assigned a value
    # ------------------------------------------------------------------
    try:
        print(pending_checkout_room)
    except NameError as ne:
        print(f"[5] NameError         >> {ne}")

    # ------------------------------------------------------------------
    # 6. TypeError
    #    Scenario: concatenate the string "Invoice #" with integer 4021
    # ------------------------------------------------------------------
    try:
        invoice_ref = "Invoice #" + 4021
    except TypeError as te:
        print(f"[6] TypeError         >> {te}")

    # ------------------------------------------------------------------
    # 7. OverflowError
    #    Scenario: an absurdly large exponential penalty calculation
    # ------------------------------------------------------------------
    try:
        import math
        penalty = math.exp(800000)
    except OverflowError as oe:
        print(f"[7] OverflowError     >> {oe}")

    # ------------------------------------------------------------------
    # 8. IOError
    #    Scenario: simulate a disk-full error while writing the bill file
    # ------------------------------------------------------------------
    try:
        raise IOError("Disk quota exceeded while writing LHMS_850009268.txt.")
    except IOError as io:
        print(f"[8] IOError           >> {io}")

    # ------------------------------------------------------------------
    # 9. ImportError
    #    Scenario: import a third-party database adapter that is not installed
    # ------------------------------------------------------------------
    try:
        import langham_db_adapter
    except ImportError as imp:
        print(f"[9] ImportError       >> {imp}")

    # ------------------------------------------------------------------
    # 10. EOFError
    #     Scenario: the nightly batch job reads stdin but the pipe is closed
    # ------------------------------------------------------------------
    try:
        raise EOFError("Batch input stream closed before room data was read.")
    except EOFError as eof:
        print(f"[10] EOFError         >> {eof}")

    # ------------------------------------------------------------------
    # 11. FileNotFoundError
    #     Scenario: attempt to open a report file that was never created
    # ------------------------------------------------------------------
    try:
        with open("LHMS_850009268_report_2024.txt", "r") as rpt:
            data = rpt.read()
    except FileNotFoundError as fnf:
        print(f"[11] FileNotFoundError>> {fnf}")

    print("=" * 65)
    print("  All 11 required exceptions demonstrated successfully.")
    print("=" * 65)


run_exception_showcase()


                TASK 2 – EXCEPTION HANDLING SHOWCASE             
[1] SyntaxError       >> expected an indented block after 'while' statement on line 1 (line 1)
[2] ValueError        >> invalid literal for int() with base 10: 'three'
[3] ZeroDivisionError >> float division by zero
[4] IndexError        >> list index out of range
[5] NameError         >> name 'pending_checkout_room' is not defined
[6] TypeError         >> can only concatenate str (not "int") to str
[7] OverflowError     >> math range error
[8] IOError           >> Disk quota exceeded while writing LHMS_850009268.txt.
[9] ImportError       >> No module named 'langham_db_adapter'
[10] EOFError         >> Batch input stream closed before room data was read.
[11] FileNotFoundError>> [Errno 2] No such file or directory: 'LHMS_850009268_report_2024.txt'
  All 11 required exceptions demonstrated successfully.
